# 9 — End to end, on CPU

Eight chapters of machinery; this is the payoff. A kernel goes from source
to typed IR to legalized ABI to **rendered Python you can read** to an
executed image — and then the loop the whole project exists for: hundreds of
frames of changing capture values, **one compile**, enforced by
`no_compile()` rather than hoped for.

New in this step: the `Registry` (surface E, v1) owning the rule packs,
backends, and both cache tiers; the Python backend (a SATELLITE — it
registers itself, zero kernel edits); and `Handle.__call__`. From this
chapter on, `import pdum.dsl` is batteries-included: the base dialect and
the backend wire themselves in, and the hand-registrations earlier chapters
performed as stand-ins are no longer needed.

In [1]:
import pdum.dsl  # batteries: base dialect + python backend register into DEFAULT
from pdum.dsl import jit, no_compile
from pdum.dsl.kernel.registry import DEFAULT


def make_disk(cx, cy, r, brightness):
    @jit()
    def disk(x, y):
        dx = x - cx
        dy = y - cy
        inside = 1.0 if dx * dx + dy * dy < r * r else 0.0
        return brightness * inside

    return disk


disk = make_disk(0.0, 0.0, 0.6, 1.0)
print("disk(0, 0)     =", disk(0.0, 0.0))
print("disk(0.9, 0.9) =", disk(0.9, 0.9))

disk(0, 0)     = 1.0
disk(0.9, 0.9) = 0.0


That is the first kernel this framework has ever *run*. Everything below is
the autopsy of what just happened — each stop on the miss path, printed.

## Stop 1: the typed, legalized IR

In [2]:
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.pack import ABI_OPS, NORMALIZE_ENV, legalize_params, plan_from_types
from pdum.dsl.kernel.printer import print_program
from pdum.dsl.kernel.rewrite import run_stage
from pdum.dsl.kernel.valuekind import BUILTINS

OPS = {**CORE_OPS, **ABI_OPS}
region = lower_handle(disk, DEFAULT.lower_rules, OPS, arg_types=(T.f64, T.f64))
plan = plan_from_types(disk.env_types, (T.f64, T.f64), BUILTINS)
abi = run_stage(run_stage(region, NORMALIZE_ENV, OPS), legalize_params(plan), OPS)
print(print_program(abi, name="disk"))

disk(%p0: f64, %p1: f64) {
  %0 = abi.slot {fmt = '<d', offset = 0, src = ('env', 0)} : f64
  %1 = abi.slot {fmt = '<d', offset = 8, src = ('env', 1)} : f64
  %2 = core.sub %p0, %1 : f64
  %3 = core.mul %2, %2 : f64
  %4 = abi.slot {fmt = '<d', offset = 16, src = ('env', 2)} : f64
  %5 = core.sub %p1, %4 : f64
  %6 = core.mul %5, %5 : f64
  %7 = core.add %3, %6 : f64
  %8 = abi.slot {fmt = '<d', offset = 24, src = ('env', 3)} : f64
  %9 = abi.slot {fmt = '<d', offset = 24, src = ('env', 3)} : f64
  %10 = core.mul %8, %9 : f64
  %11 = core.cmp %7, %10 {pred = 'lt'} : bool
  %14 = core.if %11 ({
    %12 = core.const {value = 1.0} : f64
    core.yield %12
  }, {
    %13 = core.const {value = 0.0} : f64
    core.yield %13
  }) : f64
  %15 = core.mul %0, %14 : f64
  core.yield %15
}


## Stop 2: the rendered source — read it

The backend renders that region to Python whose *calling convention is a
uniform buffer*: every input, captures and arguments alike, arrives as bytes
in staging, unpacked at the offsets the plan chose. Deliberately not the
fastest way to call Python from Python — it is the same ABI a WGSL uniform
block uses, proven on CPU first.

In [3]:
from pdum.dsl.demo.simple_shader.python import render

print(render(abi, plan, name="disk"))

from struct import unpack_from as _u

def disk(staging, leaves):
    if leaves: raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = _u('<d', staging, 0)[0]
    v1 = _u('<d', staging, 32)[0]
    v2 = _u('<d', staging, 8)[0]
    v3 = v1 - v2
    v4 = v3 * v3
    v5 = _u('<d', staging, 40)[0]
    v6 = _u('<d', staging, 16)[0]
    v7 = v5 - v6
    v8 = v7 * v7
    v9 = v4 + v8
    v10 = _u('<d', staging, 24)[0]
    v11 = _u('<d', staging, 24)[0]
    v12 = v10 * v11
    v13 = v9 < v12
    if v13:
        v14 = 1.0
        v16 = v14
    else:
        v15 = 0.0
        v16 = v15
    v17 = v0 * v16
    return v17



## Stop 3: an image

No arrays yet (ch12), so the host loops over pixels and calls the kernel
per sample — thousands of dispatches, every one a cache hit.

In [4]:
ROWS, COLS = 20, 40
shades = " ░▒▓█"
disk = make_disk(0.1, -0.1, 0.55, 1.0)
for j in range(ROWS):
    y = 1.0 - 2.0 * (j + 0.5) / ROWS
    line = ""
    for i in range(COLS):
        x = 2.0 * (i + 0.5) / COLS - 1.0
        v = disk(x, y)
        line += shades[int(v * (len(shades) - 1))]
    print(line)

                                        
                                        
                                        
                                        
                                        
                                        
                ████████████            
              ████████████████          
            ████████████████████        
           ██████████████████████       
           ██████████████████████       
           ██████████████████████       
           ██████████████████████       
            ████████████████████        
              ████████████████          
                ████████████            
                                        
                                        
                                        
                                        


## Stop 4: the loop — the thesis, enforced

Every frame builds a **fresh closure** over new values (the factory
pattern), calls it, and must hit. `no_compile()` turns "must" into an
assertion: a cold key raises `CompileForbidden` naming the differing key
component instead of silently compiling.

Note what is already true before the loop starts: the ONE compile happened
at the chapter's very first `disk(0.0, 0.0)`, and the 800-pixel image above
was 800 cache hits. The loop below adds 299 fresh closures — and zero
compiles.

In [5]:
import time

spec = DEFAULT.specializations
c0, h0 = spec.compiles, spec.hits
t0 = time.perf_counter()
with no_compile():  # a single cold key in here would RAISE, naming the differing component
    for f in range(1, 300):
        frame_disk = make_disk(0.3 * (f % 7), -0.2, 0.4 + 0.001 * f, 1.0 / f)
        frame_disk(0.25, 0.25)
dt = (time.perf_counter() - t0) / 299
print(f"total compiles for this template, whole chapter: {spec.compiles} (image included)")
print(f"new compiles in the 299-frame loop             : {spec.compiles - c0}")
print(f"hits in the loop                               : {spec.hits - h0}")
print(f"guard misses                                   : {spec.guard_misses}")
print(f"per frame: {dt * 1e6:.1f} µs — capture + fingerprint + probe + guards + pack + run")

total compiles for this template, whole chapter: 1 (image included)
new compiles in the 299-frame loop             : 0
hits in the loop                               : 299
guard misses                                   : 0
per frame: 6.9 µs — capture + fingerprint + probe + guards + pack + run


## Stop 5: the `FastRecord`, field by field

The tier-1 value is everything the hot path needs, precomputed:

In [6]:
key = DEFAULT.specializations.key_for(
    frame_disk, (BUILTINS.fingerprint(0.25),) * 2, ("demo.simple_shader.python", 1)
)
rec = DEFAULT.specializations._ready[key]
print("artifact :", rec.artifact, "(the exec'd function — it carries its listing)")
print("guards   :", [(type(h).__name__, n) for h, n, _ in rec.guards])
print("extract  :", rec.extract, "(compiled per-slot getters, ch08)")
print("plan     :", len(rec.plan.slots), "slots,", rec.plan.staging_size, "bytes staging")
print("staging  :", rec.staging.hex(), " <- reused every call; the only thing that changes")
print("launch   :", rec.launch is rec.artifact)

artifact : <function kernel at 0x1090ce090> (the exec'd function — it carries its listing)
guards   : [('cell', 'cell_contents'), ('cell', 'cell_contents'), ('cell', 'cell_contents'), ('cell', 'cell_contents')]
extract  : <function build_extractor.<locals>.<lambda> at 0x1090ce610> (compiled per-slot getters, ch08)
plan     : 6 slots, 48 bytes staging
staging  : 05eebee3e2656b3f000000000000f83f9a9999999999c9bf92ed7c3f355ee63f000000000000d03f000000000000d03f  <- reused every call; the only thing that changes
launch   : True


## Stop 6: tripping a guard

Rebind a captured local *after* decoration and the cell drifts from the
frozen env. The guard (an identity triple per capture) catches it at the
next probe: the counter climbs and the entry is rebuilt — **refuse or
recompile, never silently stale**. Semantics stay decoration-time (the env
is a snapshot; rebuild the handle — the factory pattern — to move values).

In [7]:
def sloppy():
    k = 2.0

    @jit()
    def f(x):
        return x * k

    k = 3.0  # drift: the cell no longer holds what was captured
    return f


g0 = DEFAULT.specializations.guard_misses
f = sloppy()
print("f(1.0) =", f(1.0), " (decoration-time k=2.0, not 3.0)")
print("f(1.0) =", f(1.0), " (same — and loudly counted, not silently served)")
print("guard misses:", DEFAULT.specializations.guard_misses - g0)

f(1.0) = 2.0  (decoration-time k=2.0, not 3.0)
f(1.0) = 2.0  (same — and loudly counted, not silently served)
guard misses: 1


## Stop 7: pipelines execute — and this is the house style now

`value > pipeline` dispatches through the *same* two-tier path: the fused
`Derived("pipe")` lowers via its build rule (no source!), renders as ONE
kernel, and the whole pipeline rebuilt with fresh stage values every frame
is still a single specialization.

In [8]:
from pdum.dsl.combinators import collect, op


@op
def tone(gain):
    @jit()
    def go(v):
        return v * gain

    return go


@op
def lift(b):
    @jit()
    def go(v):
        return v + b

    return go


print("0.5 > tone(2.0) | lift(0.1) | collect =", 0.5 > (tone(2.0) | lift(0.1) | collect))
c0 = DEFAULT.specializations.compiles
with no_compile():
    for f in range(1, 200):
        out = f / 200 > (tone(1.0 / f) | lift(0.01 * f) | collect)
print("fused-pipe compiles over 199 fresh pipelines:", DEFAULT.specializations.compiles - c0)

0.5 > tone(2.0) | lift(0.1) | collect = 1.1
fused-pipe compiles over 199 fresh pipelines: 0


## Stop 8: the content-addressed tier earns its keep

Two different templates with identical bodies are two *specializations* but
ONE *artifact* — and a generation bump (the coarse invalidation knob) clears
tier 1 while tier 2 survives untouched.

In [9]:
def site_a(k):
    @jit()
    def go(x):
        return x * k - 0.125

    return go


def site_b(k):  # a different def-site, an identical body
    @jit()
    def go(x):
        return x * k - 0.125

    return go


s0, a0 = DEFAULT.specializations.compiles, DEFAULT.artifacts.compiles
site_a(2.0)(3.0), site_b(5.0)(3.0)
print(f"specializations: +{DEFAULT.specializations.compiles - s0}   artifacts: +{DEFAULT.artifacts.compiles - a0}")
a1 = DEFAULT.artifacts.compiles
DEFAULT.specializations.bump_generation()
site_a(2.0)(3.0)
print(f"after bump_generation(): artifacts recompiled: +{DEFAULT.artifacts.compiles - a1} (tier 2 is generation-free)")

specializations: +2   artifacts: +1
after bump_generation(): artifacts recompiled: +0 (tier 2 is generation-free)


## The kernel budget, in the book

The step-8 exit gate requires the line-budget report *in the notebook*: the
machinery you just watched run — capture, two-tier cache, IR, rewriter,
lowering, marshaling, registry — fits in this many counted lines:

In [10]:
import json
import pathlib
import subprocess
import sys

budget = pathlib.Path(pdum.dsl.__file__).parents[3] / "scripts" / "loc_budget.py"
data = json.loads(subprocess.run([sys.executable, str(budget), "--json"],
                                 capture_output=True, text=True).stdout)
for name, f in data["files"].items():
    print(f"  {name:<14} {f['lines']:>4} / {f['cap']}")
print(f"  {'KERNEL TOTAL':<14} {data['kernel_total']:>4} / {data['kernel_cap']}")

  __init__.py       1 / 10
  api.py            7 / 50
  cache.py        160 / 165
  capture.py       75 / 85
  ir.py           109 / 150
  lower.py         94 / 170
  ops.py           86 / 110
  pack.py         169 / 175
  printer.py       46 / 80
  registry.py      89 / 110
  rewrite.py      121 / 150
  types.py         94 / 100
  valuekind.py     86 / 95
  KERNEL TOTAL   1137 / 1150


## Things to notice

- The staging hexdump in the `FastRecord` is the SAME buffer, mutated in
  place, every frame — `one value, N parameters` ended as `N values, zero
  allocations`.
- The per-frame cost includes rebuilding the closure from scratch. Phase A
  really is that cheap — that was ch02's bet, now carrying a render loop.
- The pipeline loop compiled once across 199 freshly-built pipelines:
  `Derived` identity (ch04) + the build rule (ch07) + this chapter's
  dispatch compose without any new mechanism.

## What we can't do yet

The image above was 800 Python dispatches — arrays and `core.for` (ch12)
turn it into ONE. The Python backend computes f32 as f64 and returns values
natively (the `ResultPlan` goes physical with a DPS target). Role-based
backend routing waits for a second backend — which is the next step: **WGSL
(ch10), completing M1**, where the staging buffer you have been hexdumping
becomes a real uniform block on a real GPU.